In [ ]:
import os, re, json, math, time, hashlib, random, warnings
from pathlib import Path
import numpy as np, pandas as pd, cv2
from PIL import Image, ImageOps
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
random.seed(42); np.random.seed(42)
print("OpenCV", cv2.__version__)


In [ ]:
# ================================ CONFIG ================================
INPUT_ROOT = Path("/kaggle/input")
OUT = Path("/kaggle/working/objective1"); OUT.mkdir(parents=True, exist_ok=True)
CACHE = OUT / "preprocessed_299"; CACHE.mkdir(exist_ok=True)

CLASSES = ["Melanoma", "BCC", "SCC", "MCC"]
INCLUDE_BENIGN = False
IMG_SIZE = 299

# Leave None to auto-detect a root by name; or hard-set the path.
DS = INPUT_ROOT / "datasets"          # everything is nested under here
DATASET_ROOTS = {
    "ham10000": DS,
    "isic":     DS,
    "dermacon": DS,
    "mcc":      DS,
    "partner":  None,
}

# Keep the heavy image pass tractable: cap how many images per class we analyse
# (MCC is rare, so it is effectively uncapped). Applied AFTER dedupe.
PRE_ANALYZE_CAP = {"Melanoma": 8000, "BCC": 8000, "SCC": 8000, "MCC": 10**9, "Benign": 8000}

DO_PREPROCESS_CACHE = True     # cache DullRazor + resize (needed to reuse for training)
COMPUTE_SKINTONE = True        # ITA-based skin-tone estimation
BLUR_THRESHOLD = 40.0          # Laplacian variance; below this -> rejected as blurry

TARGET_PER_CLASS = 3000        # balance TRAIN up to this via augmentation
MAX_PER_CLASS = 4000           # cap majority classes in TRAIN
VAL_RATIO, TEST_RATIO = 0.15, 0.15
print("Output ->", OUT)


In [ ]:
# ---------- label canonicalization (raw dataset label -> target class) ----------
EXACT_CODE = {"mel":"Melanoma","melanoma":"Melanoma","bcc":"BCC","scc":"SCC","akiec":"SCC",
              "mcc":"MCC","merkel":"MCC","nv":"Benign","bkl":"Benign","df":"Benign","vasc":"Benign"}
SUBSTR_RULES = [("MCC",["merkel"]),("BCC",["basal cell","basal-cell","basalcell"]),
                ("SCC",["squamous cell","squamous-cell","actinic kerat","bowen","intraepiderm","keratoacanth"]),
                ("Melanoma",["melanoma"])]
BENIGN_SUBSTR = ["nevus","nevi","naevus","benign keratos","seborrheic","dermatofibroma",
                 "vascular","angioma","lentigo","dermatitis","psoriasis","eczema","wart"]
CLASS_TO_IDX = {c:i for i,c in enumerate(CLASSES)}

def canonicalize(raw):
    if raw is None or (isinstance(raw,float) and math.isnan(raw)): return None
    s = str(raw).strip().lower(); c = EXACT_CODE.get(s)
    if c is None:
        for cls,keys in SUBSTR_RULES:
            if any(k in s for k in keys): c=cls; break
        if c is None and any(k in s for k in BENIGN_SUBSTR): c="Benign"
    if c=="Benign" and not INCLUDE_BENIGN: return None
    return c if c in CLASS_TO_IDX else None

IMG_EXT = (".jpg",".jpeg",".png",".bmp",".tif",".tiff",".webp")
def list_images(root): return [p for p in Path(root).rglob("*") if p.suffix.lower() in IMG_EXT]
def autodetect_root(*hints):
    if not INPUT_ROOT.exists(): return None
    for child in sorted(INPUT_ROOT.iterdir()):
        if any(h in child.name.lower() for h in hints): return child
    return None
def _pick(df, opts):
    for c in df.columns:
        if c.lower() in opts: return c
    return None


In [ ]:
# ---------------- STEP 2: collect + annotate -> raw manifest ----------------
records = []
def add(src, path, raw, lab, grp): records.append(dict(source=src, path=str(path), raw_label=raw, label=lab, group=grp))

def load_ham():
    root = DATASET_ROOTS["ham10000"] or autodetect_root("ham10000","ham-10000","mnist-ham")
    if not root: print("[HAM10000] skipped"); return
    root=Path(root); meta=next((p for p in root.rglob("*.csv") if "metadata" in p.name.lower()), None)
    if not meta: print("[HAM10000] no metadata csv"); return
    df=pd.read_csv(meta); idx={p.stem:p for p in list_images(root)}; n=0
    for _,r in df.iterrows():
        lab=canonicalize(r.get("dx")); p=idx.get(str(r.get("image_id")))
        if lab and p is not None:
            add("HAM10000", p, r.get("dx"), lab, str(r.get("lesion_id") or r.get("image_id"))); n+=1
    print("[HAM10000]", n)

def load_isic():
    root = DATASET_ROOTS["isic"] or autodetect_root("isic")
    if not root: print("[ISIC] skipped"); return
    root=Path(root); idx={p.stem:p for p in list_images(root)}; n=0; used=False
    for cpath in root.rglob("*.csv"):
        try: df=pd.read_csv(cpath, low_memory=False)
        except Exception: continue
        dcol=_pick(df,("diagnosis","diagnosis_1","dx","benign_malignant","disease")) or next((c for c in df.columns if "diagn" in c.lower()),None)
        idcol=_pick(df,("isic_id","image","image_id","image_name","filename","file","name","id"))
        if not dcol or not idcol: continue
        used=True; pcol=_pick(df,("patient_id","patient","lesion_id"))
        for _,r in df.iterrows():
            lab=canonicalize(r.get(dcol)); 
            if not lab: continue
            key=Path(str(r.get(idcol))).stem; p=idx.get(key)
            if p is None: continue
            grp=str(r.get(pcol)) if (pcol and pd.notna(r.get(pcol))) else key
            add("ISIC", p, r.get(dcol), lab, grp); n+=1
    if not used:
        for p in list_images(root):
            lab=canonicalize(p.parent.name)
            if lab: add("ISIC", p, p.parent.name, lab, p.stem); n+=1
    print("[ISIC]", n)

def load_dermacon():
    root = DATASET_ROOTS["dermacon"] or autodetect_root("dermacon","dermcon","derma-con")
    if not root: print("[DermaCon-IN] skipped"); return
    root=Path(root); idx={p.stem:p for p in list_images(root)}; n=0; used=False
    for cpath in root.rglob("*.csv"):
        try: df=pd.read_csv(cpath, low_memory=False)
        except Exception: continue
        dcol=_pick(df,("diagnosis","diagnosis_1","disease","label","class")) or next((c for c in df.columns if "diagn" in c.lower()),None)
        idcol=_pick(df,("image","image_id","filename","file","name","id","image_name"))
        if not dcol or not idcol: continue
        used=True
        for _,r in df.iterrows():
            lab=canonicalize(r.get(dcol)); p=idx.get(Path(str(r.get(idcol))).stem)
            if lab and p is not None: add("DermaCon-IN", p, r.get(dcol), lab, p.stem); n+=1
    if not used:
        for p in list_images(root):
            lab=canonicalize(p.parent.name)
            if lab: add("DermaCon-IN", p, p.parent.name, lab, p.stem); n+=1
    print("[DermaCon-IN]", n, "(South-Asian skin tones)")

def load_mcc():
    root = DATASET_ROOTS["mcc"] or autodetect_root("mcc","merkel")
    if not root: print("[MCC] skipped"); return
    n=0
    for p in list_images(root):
        pn=p.parent.name.lower()
        lab = ("Benign" if INCLUDE_BENIGN else None) if any(k in pn for k in ("neg","normal","benign","non")) else "MCC"
        if lab: add("MCC-Dataset", p, p.parent.name, lab, p.stem); n+=1
    print("[MCC]", n)

def load_partner():
    root = DATASET_ROOTS["partner"] or autodetect_root("partner","clinic")
    if not root: print("[Partner clinic] none attached (optional)"); return
    n=0
    for p in list_images(root):
        lab=canonicalize(p.parent.name)
        if lab: add("PartnerClinic", p, p.parent.name, lab, p.stem); n+=1
    print("[Partner clinic]", n)

for fn in (load_ham, load_isic, load_dermacon, load_mcc, load_partner):
    try: fn()
    except Exception as e: print("  !!", fn.__name__, e)

raw = pd.DataFrame(records)
assert len(raw)>0, "No images mapped. Confirm datasets are attached."
print("\nRaw annotated images:", len(raw))
print(raw.groupby(["label","source"]).size().unstack(fill_value=0))


In [ ]:
# ---------------- STEP 3a: de-duplicate (exact bytes, MD5) ----------------
def md5(path, chunk=1<<20):
    h=hashlib.md5()
    try:
        with open(path,"rb") as f:
            for b in iter(lambda: f.read(chunk), b""): h.update(b)
        return h.hexdigest()
    except Exception:
        return None
raw["md5"]=raw["path"].map(md5)
before=len(raw)
raw=raw[raw["md5"].notna()].drop_duplicates(subset="md5").reset_index(drop=True)
print(f"Removed {before-len(raw)} exact-duplicate/corrupt files; {len(raw)} remain")

# cap per class before the heavy image pass (keeps runtime tractable; MCC uncapped)
parts=[]
for c,sub in raw.groupby("label"):
    parts.append(sub.sample(min(len(sub), PRE_ANALYZE_CAP.get(c,10**9)), random_state=42))
raw=pd.concat(parts).sample(frac=1, random_state=42).reset_index(drop=True)
print("After per-class analyse cap:", len(raw), dict(raw["label"].value_counts()))


In [ ]:
# ---------------- STEP 4+5: skin tone (ITA), blur QC, preprocess cache ----------------
def dull_razor(bgr):
    gray=cv2.cvtColor(bgr,cv2.COLOR_BGR2GRAY)
    k=cv2.getStructuringElement(cv2.MORPH_CROSS,(9,9))
    bh=cv2.morphologyEx(gray,cv2.MORPH_BLACKHAT,k)
    _,mask=cv2.threshold(bh,10,255,cv2.THRESH_BINARY)
    return cv2.inpaint(bgr,mask,1,cv2.INPAINT_TELEA)

def estimate_ita(rgb):
    # ITA(deg) = arctan((L*-50)/b*) on healthy border skin (lesion assumed central).
    h,w=rgb.shape[:2]; lab=cv2.cvtColor(rgb,cv2.COLOR_RGB2Lab).astype(np.float32)
    L=lab[...,0]*100.0/255.0; b=lab[...,2]-128.0; Lraw=lab[...,0]
    m=max(4,int(0.12*min(h,w))); ring=np.zeros((h,w),bool)
    ring[:m,:]=True; ring[-m:,:]=True; ring[:,:m]=True; ring[:,-m:]=True
    skin=ring & (Lraw>40) & (Lraw<245)              # drop hair (dark) & specular (bright)
    if skin.sum()<50: skin=(Lraw>40)&(Lraw<245)
    if skin.sum()<10: return float("nan")
    return float(np.degrees(np.arctan2(np.median(L[skin])-50.0, np.median(b[skin])+1e-6)))

def ita_band(ita):
    if np.isnan(ita): return "Unknown"
    if ita>55: return "Very light (I)"
    if ita>41: return "Light (II)"
    if ita>28: return "Intermediate (III)"
    if ita>10: return "Tan (IV)"
    if ita>-30: return "Brown (V)"
    return "Dark (VI)"

def cache_path_for(p): return CACHE/(re.sub(r"[^A-Za-z0-9]+","_",str(p))[-120:]+".npy")

blur_l, ita_l, ok_l = [], [], []
t0=time.time()
for i,p in enumerate(raw["path"].tolist()):
    try:
        pil=ImageOps.exif_transpose(Image.open(p).convert("RGB")); rgb=np.array(pil)
    except Exception:
        blur_l.append(np.nan); ita_l.append(np.nan); ok_l.append(False); continue
    gray=cv2.cvtColor(rgb,cv2.COLOR_RGB2GRAY)
    blur_l.append(float(cv2.Laplacian(gray,cv2.CV_64F).var()))
    ita_l.append(estimate_ita(rgb) if COMPUTE_SKINTONE else np.nan)
    if DO_PREPROCESS_CACHE:
        bgr=dull_razor(cv2.cvtColor(rgb,cv2.COLOR_RGB2BGR))
        bgr=cv2.resize(bgr,(IMG_SIZE,IMG_SIZE),interpolation=cv2.INTER_CUBIC)
        np.save(cache_path_for(p), cv2.cvtColor(bgr,cv2.COLOR_BGR2RGB))
    ok_l.append(True)
    if i%2000==0: print(f"  {i}/{len(raw)}  ({time.time()-t0:.0f}s)")
raw["blur"]=blur_l; raw["ita"]=ita_l; raw["ok"]=ok_l
raw["skin_band"]=raw["ita"].map(ita_band)

corrupt=(~raw["ok"]).sum(); blurry=((raw["ok"]) & (raw["blur"]<BLUR_THRESHOLD)).sum()
raw=raw[(raw["ok"]) & (raw["blur"]>=BLUR_THRESHOLD)].reset_index(drop=True)
print(f"\nQC: dropped {corrupt} corrupt + {blurry} blurry -> {len(raw)} clean images")
print("done in", round(time.time()-t0), "s")


In [ ]:
# ---------------- STEP 6a: leakage-safe grouped split ----------------
from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
def grouped_split(frame, val_ratio, test_ratio, seed=42):
    y=frame["label"].map(CLASS_TO_IDX).values; g=frame["group"].values
    try:
        s=StratifiedGroupKFold(n_splits=max(2,round(1/test_ratio)),shuffle=True,random_state=seed)
        tr,te=next(s.split(frame,y,g)); a,b=frame.iloc[tr].copy(),frame.iloc[te].copy()
        s2=StratifiedGroupKFold(n_splits=max(2,round(1/(val_ratio/(1-test_ratio)))),shuffle=True,random_state=seed)
        x,v=next(s2.split(a,a["label"].map(CLASS_TO_IDX).values,a["group"].values))
        return a.iloc[x].copy(), a.iloc[v].copy(), b
    except Exception as e:
        print("fallback GroupShuffleSplit:",e)
        gss=GroupShuffleSplit(1,test_size=test_ratio,random_state=seed); tr,te=next(gss.split(frame,y,g))
        a,b=frame.iloc[tr].copy(),frame.iloc[te].copy()
        g2=GroupShuffleSplit(1,test_size=val_ratio/(1-test_ratio),random_state=seed); x,v=next(g2.split(a,a["label"],a["group"]))
        return a.iloc[x].copy(), a.iloc[v].copy(), b

df_train, df_val, df_test = grouped_split(raw, VAL_RATIO, TEST_RATIO)
for d,n in [(df_train,"train"),(df_val,"val"),(df_test,"test")]: d["split"]=n
print("sizes:", len(df_train), len(df_val), len(df_test))


In [1]:
from pathlib import Path

datasets = [
    "/kaggle/input/datasets/abhig888/dermacon-in",
    "/kaggle/input/datasets/hiro002/dermacon-in-dataset",
    "/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled",
    "/kaggle/input/datasets/quantumcoders05/mcc-dataset",
    "/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic",
    "/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000",
    "/kaggle/input/datasets/ahmedxc4/skin-ds",
    "/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"
]

for d in datasets:
    print("\n" + "="*80)
    print(d)

    root = Path(d)

    if not root.exists():
        print("❌ NOT FOUND")
        continue

    for item in sorted(root.iterdir()):
        if item.is_dir():
            print("📁", item.name)
        else:
            print("📄", item.name)


/kaggle/input/datasets/abhig888/dermacon-in
📁 DATASET
📄 Infer_CBM_MC.ipynb
📄 Infer_Swin_MC.ipynb
📁 METADATA
📄 Metadata_schema.md
📄 README.md
📁 checkpoints
📄 train_CBM_SC_MC_type1.py
📄 train_CBM_SC_MC_type2.py
📄 train_cbm_mc.py
📄 train_swin_mc.py

/kaggle/input/datasets/hiro002/dermacon-in-dataset
📁 DATASET
📄 Infer_CBM_MC.ipynb
📄 Infer_Swin_MC.ipynb
📁 METADATA
📄 Metadata_schema.md
📄 README.md
📁 checkpoints
📄 train_CBM_SC_MC_type1.py
📄 train_CBM_SC_MC_type2.py
📄 train_cbm_mc.py
📄 train_swin_mc.py

/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled
📁 ISIC_Labelled

/kaggle/input/datasets/quantumcoders05/mcc-dataset
❌ NOT FOUND

/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic
📁 Skin cancer ISIC The International Skin Imaging Collaboration
📁 skin cancer isic the international skin imaging collaboration

/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
📁 HAM10000_images_part_1
📁 HAM10000_images_part_2
📄 HAM10000_metadata.csv
📁 ham10000_images_

In [2]:
from pathlib import Path

# Dataset roots that were found
dataset_roots = {
    "dermacon_1": Path("/kaggle/input/datasets/abhig888/dermacon-in"),
    "dermacon_2": Path("/kaggle/input/datasets/hiro002/dermacon-in-dataset"),
    "isic_labelled": Path("/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled"),
    "isic_9class": Path("/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"),
    "ham10000": Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"),
    "skin_ds": Path("/kaggle/input/datasets/ahmedxc4/skin-ds"),
    "all_isic": Path("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"),
}

# File extensions we care about
image_ext = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
data_ext = {".csv", ".json", ".txt", ".xlsx"}

for name, root in dataset_roots.items():

    print("\n" + "=" * 90)
    print(f"DATASET: {name}")
    print(f"PATH:    {root}")

    if not root.exists():
        print("STATUS: NOT FOUND")
        continue

    image_count = 0
    data_files = []

    # Look through files safely
    for file in root.rglob("*"):

        if not file.is_file():
            continue

        if file.suffix.lower() in image_ext:
            image_count += 1

        elif file.suffix.lower() in data_ext:
            data_files.append(file)

    print(f"IMAGE COUNT: {image_count}")

    print("\nDATA / METADATA FILES:")
    if data_files:
        for file in sorted(data_files)[:30]:
            print("  ", file.relative_to(root))
    else:
        print("   None found")

    print("\nTOP-LEVEL FOLDERS:")
    folders = [x for x in root.iterdir() if x.is_dir()]

    if folders:
        for folder in sorted(folders):
            print("  📁", folder.name)
    else:
        print("   None")


DATASET: dermacon_1
PATH:    /kaggle/input/datasets/abhig888/dermacon-in
IMAGE COUNT: 5450

DATA / METADATA FILES:
   METADATA/Skin_Metadata-1.csv
   METADATA/test_split-1.csv
   METADATA/train_split-1.csv

TOP-LEVEL FOLDERS:
  📁 DATASET
  📁 METADATA
  📁 checkpoints

DATASET: dermacon_2
PATH:    /kaggle/input/datasets/hiro002/dermacon-in-dataset
IMAGE COUNT: 5450

DATA / METADATA FILES:
   METADATA/Skin_Metadata.csv
   METADATA/test_split.csv
   METADATA/train_split.csv

TOP-LEVEL FOLDERS:
  📁 DATASET
  📁 METADATA
  📁 checkpoints

DATASET: isic_labelled
PATH:    /kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled


KeyboardInterrupt: 

In [3]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in")

print("DATASET:", root)
print("\nInside dataset:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

DATASET: /kaggle/input/datasets/abhig888/dermacon-in

Inside dataset:

📁 DATASET
📄 Infer_CBM_MC.ipynb
📄 Infer_Swin_MC.ipynb
📁 METADATA
📄 Metadata_schema.md
📄 README.md
📁 checkpoints
📄 train_CBM_SC_MC_type1.py
📄 train_CBM_SC_MC_type2.py
📄 train_cbm_mc.py
📄 train_swin_mc.py


In [4]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in/DATASET")

print("DATASET FOLDER:", root)
print("\nInside DATASET:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

DATASET FOLDER: /kaggle/input/datasets/abhig888/dermacon-in/DATASET

Inside DATASET:

📁 DATASET_0
📁 DATASET_1


In [1]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in/DATASET")

print("DATASET FOLDER:", root)
print("\nInside DATASET:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

DATASET FOLDER: /kaggle/input/datasets/abhig888/dermacon-in/DATASET

Inside DATASET:

📁 DATASET_0
📁 DATASET_1


In [2]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_0")

print("DATASET_0:", root)
print("\nInside DATASET_0:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

DATASET_0: /kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_0

Inside DATASET_0:

📁 DATASET_0


In [3]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_0/DATASET_0")

print("INNER DATASET_0:", root)
print("\nInside INNER DATASET_0:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

INNER DATASET_0: /kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_0/DATASET_0

Inside INNER DATASET_0:

📄 IMG_0002.jpg
📄 IMG_0005.jpg
📄 IMG_0009.jpg
📄 IMG_0010.jpg
📄 IMG_0011.jpg
📄 IMG_0015.jpg
📄 IMG_0017.jpg
📄 IMG_0019.jpg
📄 IMG_0031.jpg
📄 IMG_0033.jpg
📄 IMG_0034.jpg
📄 IMG_0036.jpg
📄 IMG_0040.jpg
📄 IMG_0044.jpg
📄 IMG_0045.jpg
📄 IMG_0048.jpg
📄 IMG_0051.jpg
📄 IMG_0053.jpg
📄 IMG_0055.jpg
📄 IMG_0056.jpg
📄 IMG_0057.jpg
📄 IMG_0058.jpg
📄 IMG_0060.jpg
📄 IMG_0063.jpg
📄 IMG_0064.jpg
📄 IMG_0065.jpg
📄 IMG_0068.jpg
📄 IMG_0072.jpg
📄 IMG_0073.jpg
📄 IMG_0075.jpg
📄 IMG_0077.jpg
📄 IMG_0079.jpg
📄 IMG_0081.jpg
📄 IMG_0082.jpg
📄 IMG_0083.jpg
📄 IMG_0084.jpg
📄 IMG_0087.jpg
📄 IMG_0089.jpg
📄 IMG_0090.jpg
📄 IMG_0096.jpg
📄 IMG_0097.jpg
📄 IMG_0101.jpg
📄 IMG_0102.jpg
📄 IMG_0103.jpg
📄 IMG_0104.jpg
📄 IMG_0108.jpg
📄 IMG_0109.jpg
📄 IMG_0110.jpg
📄 IMG_0111.jpg
📄 IMG_0114.jpg
📄 IMG_0115.jpg
📄 IMG_0117.jpg
📄 IMG_0120.jpg
📄 IMG_0123.jpg
📄 IMG_0125.jpg
📄 IMG_0126.jpg
📄 IMG_0127.jpg
📄 IMG_0129.jpg
📄 IMG_0131.jpg


In [4]:
from pathlib import Path

root = Path("/kaggle/input/datasets/abhig888/dermacon-in/METADATA")

print("METADATA:", root)
print("\nInside METADATA:\n")

for item in sorted(root.iterdir()):
    if item.is_dir():
        print("📁", item.name)
    else:
        print("📄", item.name)

METADATA: /kaggle/input/datasets/abhig888/dermacon-in/METADATA

Inside METADATA:

📄 Skin_Metadata-1.csv
📄 test_split-1.csv
📄 train_split-1.csv


In [5]:
import pandas as pd
from pathlib import Path

csv_path = Path(
    "/kaggle/input/datasets/abhig888/dermacon-in/METADATA/Skin_Metadata-1.csv"
)

df = pd.read_csv(csv_path)

print("Shape:", df.shape)

print("\nColumns:")
for col in df.columns:
    print(" -", col)

print("\nFirst 5 rows:")
display(df.head())

Shape: (5450, 14)

Columns:
 - Age
 - Body_part
 - Descriptors
 - Confidence
 - Disease_label
 - Fitzpatrick
 - Gradability
 - Image_name
 - Main_class
 - Monk_skin_tone
 - Quality
 - Sex
 - Sub_class
 - Subject_ID

First 5 rows:


,Age,Body_part,Descriptors,Confidence,Disease_label,Fitzpatrick,Gradability,Image_name,Main_class,Monk_skin_tone,Quality,Sex,Sub_class,Subject_ID
0,10 - 20,"Groin (Inguinal Region),Lower Extremities Thig...","Erythema, Pigmented, Plaque",5,Steroid Modified Tinea,FST 3,Yes,IMG_0002.jpg,Infectious Disorders,MST 5,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00001
1,10 - 20,"Groin (Inguinal Region),Lower Extremities Thig...","Erythema, Pigmented, Plaque",5,Steroid Modified Tinea,FST 3,Yes,IMG_0005.jpg,Infectious Disorders,MST 5,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00001
2,40 - 60,Head Cheeks,Papule,5,Acne,FST 4,Yes,IMG_0009.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002
3,40 - 60,Head Cheeks,Papule,5,Acne,FST 4,Yes,IMG_0010.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002
4,40 - 60,"Head Cheeks,Head Nose,Head Lips",Papule,5,Acne,FST 4,Yes,IMG_0011.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002


In [6]:
# Check disease labels available in DermaCon

cancer_keywords = [
    "melanoma",
    "basal cell",
    "squamous cell",
    "merkel"
]

disease_counts = (
    df["Disease_label"]
    .astype(str)
    .str.strip()
    .value_counts()
)

print("Disease labels matching our target cancer classes:\n")

found = False

for disease, count in disease_counts.items():
    disease_lower = disease.lower()

    if any(keyword in disease_lower for keyword in cancer_keywords):
        print(f"{disease}  -->  {count} images")
        found = True

if not found:
    print("❌ No target cancer labels found.")

Disease labels matching our target cancer classes:

Basal Cell Carcinoma  -->  10 images
Squamous Cell Carcinoma  -->  5 images
Melanoma  -->  2 images


In [7]:
print("Total unique disease labels:", df["Disease_label"].nunique())

print("\nAll disease labels and counts:\n")

display(
    df["Disease_label"]
    .astype(str)
    .str.strip()
    .value_counts()
    .to_frame("Image_Count")
)

Total unique disease labels: 245

All disease labels and counts:



,Image_Count
Disease_label,
Vitiligo,601
Tinea Cruris,545
Tinea Corporis,323
Scabies,183
Acne,181
...,...
Perioral dermatitis,1
Kerion,1
Myxoid Cyst,1


In [8]:
# Check cancer-related records using all relevant label columns

keywords = [
    "melanoma",
    "basal cell",
    "squamous cell",
    "merkel"
]

mask = (
    df["Disease_label"].astype(str).str.lower().str.contains("|".join(keywords), regex=True)
    | df["Main_class"].astype(str).str.lower().str.contains("|".join(keywords), regex=True)
    | df["Sub_class"].astype(str).str.lower().str.contains("|".join(keywords), regex=True)
)

cancer_records = df.loc[
    mask,
    [
        "Image_name",
        "Disease_label",
        "Main_class",
        "Sub_class",
        "Age",
        "Body_part",
        "Fitzpatrick",
        "Monk_skin_tone",
        "Sex",
        "Subject_ID"
    ]
]

print("Cancer-related records found:", len(cancer_records))

display(cancer_records)

Cancer-related records found: 17


,Image_name,Disease_label,Main_class,Sub_class,Age,Body_part,Fitzpatrick,Monk_skin_tone,Sex,Subject_ID
76,IMG_0166.jpg,Squamous Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,40 - 60,Trunk Abdomen,FST 3,MST 5,Female,SUB00041
802,IMG_1834.jpg,Squamous Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,60 - 80,Upper Extremities Shoulders,FST 5,MST 7,Male,SUB00431
873,IMG_20190712_105158.jpg,Squamous Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,80 - 100,Head Nose,FST 3,MST 5,Female,SUB01947
877,IMG_20190816_122607_1.jpg,Squamous Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,80 - 100,Head Nose,FST 4,MST 6,Female,SUB01947
1039,IMG_20230105_203256.jpg,Basal Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,40 - 60,Head Nose,FST 5,MST 7,Male,SUB02498
1040,IMG_20230105_203258_1.jpg,Basal Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,40 - 60,Head Nose,FST 5,MST 7,Male,SUB02498
1491,IMG_20230720_122930_BURST3_1.jpg,Basal Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,60 - 80,"Head Cheeks,Head Forehead, Head Nose",FST 4,MST 6,Female,SUB02239
1626,IMG_20231117_140137_1.jpg,Squamous Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,80 - 100,Head Temples,FST 5,MST 7,Female,SUB02874
1849,IMG_20240611_132130_1.jpg,Basal Cell Carcinoma,Neoplasms and tumors,Neoplasms and tumors,60 - 80,Head Cheeks,FST 5,MST 7,Male,SUB00489
2176,IMG_2807.jpg,Melanoma,Neoplasms and tumors,Neoplasms and tumors,20 - 40,"Back,LowerBack (Lumbus)",FST 3,MST 4,Female,SUB00622


In [9]:
from pathlib import Path

image_root = Path(
    "/kaggle/input/datasets/abhig888/dermacon-in/DATASET"
)

# Get all image filenames inside DATASET
image_files = {}

for p in image_root.rglob("*"):
    if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp", ".webp"}:
        image_files[p.name.lower()] = str(p)

# Check metadata cancer images against actual files
check = cancer_records.copy()

check["file_found"] = check["Image_name"].astype(str).str.lower().map(
    lambda x: x in image_files
)

print("Total cancer metadata records:", len(check))
print("Images found:", check["file_found"].sum())
print("Images NOT found:", (~check["file_found"]).sum())

print("\nMissing images:")
display(check.loc[~check["file_found"], ["Image_name", "Disease_label"]])

Total cancer metadata records: 17
Images found: 17
Images NOT found: 0

Missing images:


,Image_name,Disease_label


In [10]:
from pathlib import Path
import pandas as pd

metadata_root = Path(
    "/kaggle/input/datasets/abhig888/dermacon-in/METADATA"
)

for filename in ["train_split-1.csv", "test_split-1.csv"]:

    path = metadata_root / filename
    temp = pd.read_csv(path)

    print("\n" + "=" * 70)
    print(filename)
    print("Shape:", temp.shape)

    print("\nColumns:")
    for col in temp.columns:
        print(" -", col)

    print("\nFirst 5 rows:")
    display(temp.head())


train_split-1.csv
Shape: (4399, 108)

Columns:
 - Age
 - Body_part_Armpits (Axillary Region)
 - Body_part_Back
 - Body_part_Back of the Knees (Popliteal Region)
 - Body_part_Breasts (Mammary Region)
 - Body_part_Calves (Sural Region)
 - Body_part_Genital Area (Pubic Region)
 - Body_part_Groin (Inguinal Region)
 - Body_part_Head Cheeks
 - Body_part_Head Chin
 - Body_part_Head Ears
 - Body_part_Head Eye
 - Body_part_Head Forehead
 - Body_part_Head Lips
 - Body_part_Head Nose
 - Body_part_Head Scalp
 - Body_part_Head Temples
 - Body_part_Lower Extremities Ankles (Tarsal Region)
 - Body_part_Lower Extremities Feet (Pedal Region)
 - Body_part_Lower Extremities Feet (Pedal Region) Toes (Digits)
 - Body_part_Lower Extremities Hips (Coxal Region)
 - Body_part_Lower Extremities Knees (Patellar Region)
 - Body_part_Lower Extremities Lower Legs
 - Body_part_Lower Extremities Lower Legs (Crural Region)
 - Body_part_Lower Extremities Soles (Plantar Region)
 - Body_part_Lower Extremities Thighs (Fe

,Age,Body_part_Armpits (Axillary Region),Body_part_Back,Body_part_Back of the Knees (Popliteal Region),Body_part_Breasts (Mammary Region),Body_part_Calves (Sural Region),Body_part_Genital Area (Pubic Region),Body_part_Groin (Inguinal Region),Body_part_Head Cheeks,Body_part_Head Chin,...,Disease_label,Fitzpatrick,Gradability,Image_name,Main_class,Monk_skin_tone,Quality,Sex,Sub_class,Subject_ID
0,10 - 20,0,0,0,0,0,0,1,0,0,...,Steroid Modified Tinea,FST 3,Yes,IMG_0002.jpg,Infectious Disorders,MST 5,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00001
1,10 - 20,0,0,0,0,0,0,1,0,0,...,Steroid Modified Tinea,FST 3,Yes,IMG_0005.jpg,Infectious Disorders,MST 5,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00001
2,40 - 60,0,0,0,0,0,0,0,1,0,...,Acne,FST 4,Yes,IMG_0009.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002
3,40 - 60,0,0,0,0,0,0,0,1,0,...,Acne,FST 4,Yes,IMG_0010.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002
4,40 - 60,0,0,0,0,0,0,0,1,0,...,Acne,FST 4,Yes,IMG_0011.jpg,Skin Appendages Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Sebacious Glands and Acneiform Disorders,SUB00002



test_split-1.csv
Shape: (1051, 108)

Columns:
 - Age
 - Body_part_Armpits (Axillary Region)
 - Body_part_Back
 - Body_part_Back of the Knees (Popliteal Region)
 - Body_part_Breasts (Mammary Region)
 - Body_part_Calves (Sural Region)
 - Body_part_Genital Area (Pubic Region)
 - Body_part_Groin (Inguinal Region)
 - Body_part_Head Cheeks
 - Body_part_Head Chin
 - Body_part_Head Ears
 - Body_part_Head Eye
 - Body_part_Head Forehead
 - Body_part_Head Lips
 - Body_part_Head Nose
 - Body_part_Head Scalp
 - Body_part_Head Temples
 - Body_part_Lower Extremities Ankles (Tarsal Region)
 - Body_part_Lower Extremities Feet (Pedal Region)
 - Body_part_Lower Extremities Feet (Pedal Region) Toes (Digits)
 - Body_part_Lower Extremities Hips (Coxal Region)
 - Body_part_Lower Extremities Knees (Patellar Region)
 - Body_part_Lower Extremities Lower Legs
 - Body_part_Lower Extremities Lower Legs (Crural Region)
 - Body_part_Lower Extremities Soles (Plantar Region)
 - Body_part_Lower Extremities Thighs (Fem

,Age,Body_part_Armpits (Axillary Region),Body_part_Back,Body_part_Back of the Knees (Popliteal Region),Body_part_Breasts (Mammary Region),Body_part_Calves (Sural Region),Body_part_Genital Area (Pubic Region),Body_part_Groin (Inguinal Region),Body_part_Head Cheeks,Body_part_Head Chin,...,Disease_label,Fitzpatrick,Gradability,Image_name,Main_class,Monk_skin_tone,Quality,Sex,Sub_class,Subject_ID
0,20 - 40,0,0,0,0,0,0,0,0,1,...,Tinea Faciei,FST 4,Yes,IMG_0017.jpg,Infectious Disorders,MST 7,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00004
1,20 - 40,0,0,0,0,0,0,0,0,1,...,Tinea Faciei,FST 4,Yes,IMG_0019.jpg,Infectious Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Male,Infectious skin conditions -Fungal,SUB00004
2,40 - 60,0,0,0,1,0,0,0,0,0,...,Pruritic dermatitis,FST 4,Yes,IMG_0044.jpg,Inflammatory Disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Female,Inflammatory skin diseases (Eczema and Dermati...,SUB00008
3,20 - 40,0,0,0,0,0,0,0,0,0,...,Cheilitis,FST 4,Yes,IMG_0084.jpg,Other skin disorders,MST 6,"Yes, the quality is sufficient for me or anoth...",Male,Other skin disorders,SUB00020
4,40 - 60,0,0,0,0,0,0,0,1,0,...,Tinea Faciei,FST 4,Yes,IMG_0087.jpg,Infectious Disorders,MST 7,"Yes, the quality is sufficient for me or anoth...",Female,Infectious skin conditions -Fungal,SUB00021


In [11]:
# ============================================================
# STEP 11 — DermaCon train/test split validation
# ============================================================

train_path = metadata_root / "train_split-1.csv"
test_path = metadata_root / "test_split-1.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# ------------------------------------------------------------
# 1. Cancer class counts
# ------------------------------------------------------------

target_keywords = {
    "Melanoma": "melanoma",
    "BCC": "basal cell carcinoma",
    "SCC": "squamous cell carcinoma",
    "MCC": "merkel"
}

print("=" * 70)
print("CANCER COUNTS")
print("=" * 70)

for class_name, keyword in target_keywords.items():

    train_count = train_df["Disease_label"].astype(str).str.lower().str.contains(
        keyword, regex=False
    ).sum()

    test_count = test_df["Disease_label"].astype(str).str.lower().str.contains(
        keyword, regex=False
    ).sum()

    print(f"{class_name:10s} | Train: {train_count:4d} | Test: {test_count:4d}")


# ------------------------------------------------------------
# 2. Subject-level leakage check
# ------------------------------------------------------------

train_subjects = set(train_df["Subject_ID"].astype(str))
test_subjects = set(test_df["Subject_ID"].astype(str))

overlap_subjects = train_subjects.intersection(test_subjects)

print("\n" + "=" * 70)
print("SUBJECT-LEVEL LEAKAGE CHECK")
print("=" * 70)

print("Unique train subjects:", len(train_subjects))
print("Unique test subjects :", len(test_subjects))
print("Subjects in BOTH     :", len(overlap_subjects))

if len(overlap_subjects) == 0:
    print("✅ No Subject_ID overlap between train and test.")
else:
    print("⚠️ Subject_ID overlap detected.")


# ------------------------------------------------------------
# 3. Show overlapping subjects if any
# ------------------------------------------------------------

if len(overlap_subjects) > 0:
    print("\nOverlapping Subject_IDs:")
    print(sorted(list(overlap_subjects))[:50])

CANCER COUNTS
Melanoma   | Train:    2 | Test:    0
BCC        | Train:    6 | Test:    4
SCC        | Train:    3 | Test:    2
MCC        | Train:    0 | Test:    0

SUBJECT-LEVEL LEAKAGE CHECK
Unique train subjects: 2396
Unique test subjects : 606
Subjects in BOTH     : 0
✅ No Subject_ID overlap between train and test.


In [12]:
# ============================================================
# STEP 12 — Inspect DermaCon DATASET_1
# ============================================================

dataset1_root = dermacon_root / "DATASET" / "DATASET_1"

print("=" * 70)
print("DERMACON DATASET_1")
print("=" * 70)

if dataset1_root.exists():
    print("Path:", dataset1_root)

    items = list(dataset1_root.iterdir())

    print("Number of direct items:", len(items))

    for item in items[:30]:
        if item.is_dir():
            print("[FOLDER]", item.name)
        else:
            print("[FILE]  ", item.name)

else:
    print("❌ DATASET_1 path not found:")
    print(dataset1_root)

NameError: name 'dermacon_root' is not defined

In [13]:
from pathlib import Path

dermacon_root = Path("/kaggle/input/datasets/abhig888/dermacon-in")
dataset1_root = dermacon_root / "DATASET" / "DATASET_1"

print("=" * 70)
print("DERMACON DATASET_1")
print("=" * 70)

print("Path:", dataset1_root)
print("Exists:", dataset1_root.exists())

if dataset1_root.exists():
    items = list(dataset1_root.iterdir())

    print("Number of direct items:", len(items))
    print()

    for item in items[:30]:
        if item.is_dir():
            print("[FOLDER]", item.name)
        else:
            print("[FILE]  ", item.name)
else:
    print("❌ DATASET_1 path not found")

DERMACON DATASET_1
Path: /kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_1
Exists: True
Number of direct items: 1

[FOLDER] DATASET_1


In [14]:
# ============================================================
# STEP 13 — Inspect nested DATASET_1
# ============================================================

nested_root = dataset1_root / "DATASET_1"

print("=" * 70)
print("NESTED DATASET_1")
print("=" * 70)

print("Path:", nested_root)
print("Exists:", nested_root.exists())

if nested_root.exists():
    items = list(nested_root.iterdir())

    print("Number of direct items:", len(items))
    print()

    for item in items[:50]:
        if item.is_dir():
            print("[FOLDER]", item.name)
        else:
            print("[FILE]  ", item.name)
else:
    print("❌ Nested DATASET_1 not found")

NESTED DATASET_1
Path: /kaggle/input/datasets/abhig888/dermacon-in/DATASET/DATASET_1/DATASET_1
Exists: True
Number of direct items: 2824

[FILE]   IMG_E3472_2.jpg
[FILE]   IMG_20240214_120444_1.jpg
[FILE]   IMG_20230713_135805_1.jpg
[FILE]   IMG_9304_1.jpg
[FILE]   IMG_9816_1.jpg
[FILE]   IMG_8939_1.jpg
[FILE]   IMG_20240614_120217_1.jpg
[FILE]   IMG20240927105354.jpg
[FILE]   IMG_6574_1.jpg
[FILE]   IMG20241114120454.jpg
[FILE]   IMG_6037_1.jpg
[FILE]   IMG_7625_1.jpg
[FILE]   IMG_7052_1.jpg
[FILE]   IMG20241019123243.jpg
[FILE]   IMG_20230819_145056_1_1_1.jpg
[FILE]   IMG_20230322_191559_1.jpg
[FILE]   IMG_8327.jpg
[FILE]   IMG_20230117_194959_1.jpg
[FILE]   IMG_20231128_200635_1.jpg
[FILE]   IMG_20221212_194232_1_1.jpg
[FILE]   IMG_20230103_132727_1_1.jpg
[FILE]   IMG20240927163049.jpg
[FILE]   IMG_9397_1.jpg
[FILE]   IMG_20230703_150633_1.jpg
[FILE]   IMG_20230614_122246_1.jpg
[FILE]   IMG_7790_1.jpg
[FILE]   IMG_7947.jpg
[FILE]   IMG_20240214_115923_1.jpg
[FILE]   IMG_9037_1.jpg
[

In [15]:
# ============================================================
# STEP 14 — Match DATASET_1 images with DermaCon metadata
# ============================================================

import pandas as pd
from pathlib import Path

metadata_file = dermacon_root / "METADATA" / "Skin_Metadata-1.csv"

meta = pd.read_csv(metadata_file)

# Image names from DATASET_1
image_files = [
    p for p in nested_root.iterdir()
    if p.is_file() and p.suffix.lower() in [".jpg", ".jpeg", ".png"]
]

image_names = {p.name.lower() for p in image_files}

# Metadata image names
meta["Image_name_clean"] = (
    meta["Image_name"]
    .astype(str)
    .str.strip()
    .str.lower()
)

matched = meta[meta["Image_name_clean"].isin(image_names)].copy()

print("=" * 70)
print("DATASET_1 ↔ METADATA MATCHING")
print("=" * 70)

print("DATASET_1 image files :", len(image_files))
print("Metadata total rows   :", len(meta))
print("Matched metadata rows :", len(matched))

print("\n" + "=" * 70)
print("CANCER LABELS AMONG MATCHED IMAGES")
print("=" * 70)

cancer_keywords = {
    "Melanoma": "melanoma",
    "BCC": "basal cell carcinoma",
    "SCC": "squamous cell carcinoma",
    "MCC": "merkel"
}

for class_name, keyword in cancer_keywords.items():
    count = matched["Disease_label"].astype(str).str.lower().str.contains(
        keyword, regex=False
    ).sum()
    print(f"{class_name:10s}: {count}")

print("\n" + "=" * 70)
print("SAMPLE MATCHED LABELS")
print("=" * 70)

if len(matched) > 0:
    print(
        matched[["Image_name", "Disease_label"]]
        .head(20)
        .to_string(index=False)
    )
else:
    print("❌ No metadata matches found.")

DATASET_1 ↔ METADATA MATCHING
DATASET_1 image files : 2824
Metadata total rows   : 5450
Matched metadata rows : 2824

CANCER LABELS AMONG MATCHED IMAGES
Melanoma  : 0
BCC       : 9
SCC       : 3
MCC       : 0

SAMPLE MATCHED LABELS
               Image_name           Disease_label
  IMG_20190503_210009.jpg              Hemangioma
  IMG_20190506_200330.jpg                Furuncle
  IMG_20190508_192238.jpg             Acrochordon
  IMG_20190508_193612.jpg             Acrochordon
  IMG_20190511_181817.jpg               Psoriasis
  IMG_20190511_181852.jpg               Psoriasis
IMG_20190614_182343_1.jpg         Photodermatitis
IMG_20190614_182352_1.jpg         Photodermatitis
IMG_20190614_182401_2.jpg         Photodermatitis
IMG_20190706_191716_1.jpg           Herpes Zoster
  IMG_20190709_104403.jpg              Hemangioma
  IMG_20190712_105158.jpg Squamous Cell Carcinoma
  IMG_20190716_182650.jpg   Cutaneous amyloidosis
  IMG_20190716_201400.jpg           Herpes Zoster
  IMG_20190722_184

In [16]:
# ============================================================
# STEP 15 — Inspect 2nd DermaCon Dataset
# ============================================================

from pathlib import Path

dermacon2_root = Path("/kaggle/input/datasets/hiro002/dermacon-in-dataset")

print("=" * 70)
print("DERMACON-IN-DATASET")
print("=" * 70)

print("Path:", dermacon2_root)
print("Exists:", dermacon2_root.exists())

if dermacon2_root.exists():
    items = list(dermacon2_root.iterdir())

    print("Number of direct items:", len(items))
    print()

    for item in items[:50]:
        if item.is_dir():
            print("[FOLDER]", item.name)
        else:
            print("[FILE]  ", item.name)
else:
    print("❌ Dataset path not found")

DERMACON-IN-DATASET
Path: /kaggle/input/datasets/hiro002/dermacon-in-dataset
Exists: True
Number of direct items: 11

[FOLDER] METADATA
[FILE]   train_CBM_SC_MC_type2.py
[FILE]   Infer_Swin_MC.ipynb
[FOLDER] DATASET
[FOLDER] checkpoints
[FILE]   Infer_CBM_MC.ipynb
[FILE]   README.md
[FILE]   train_CBM_SC_MC_type1.py
[FILE]   train_cbm_mc.py
[FILE]   Metadata_schema.md
[FILE]   train_swin_mc.py


In [17]:
# ============================================================
# STEP 16 — Inspect 2nd DermaCon DATASET
# ============================================================

dataset2_root = dermacon2_root / "DATASET"

print("=" * 70)
print("DERMACON-IN-DATASET / DATASET")
print("=" * 70)

print("Path:", dataset2_root)
print("Exists:", dataset2_root.exists())

if dataset2_root.exists():
    items = list(dataset2_root.iterdir())

    print("Number of direct items:", len(items))
    print()

    for item in items[:50]:
        if item.is_dir():
            print("[FOLDER]", item.name)
        else:
            print("[FILE]  ", item.name)
else:
    print("❌ DATASET folder not found")

DERMACON-IN-DATASET / DATASET
Path: /kaggle/input/datasets/hiro002/dermacon-in-dataset/DATASET
Exists: True
Number of direct items: 2

[FOLDER] DATASET_0
[FOLDER] DATASET_1


In [18]:
# ============================================================
# STEP 17 — Inspect DATASET_0 and DATASET_1
# ============================================================

dataset0_root = dataset2_root / "DATASET_0"
dataset1_root_2 = dataset2_root / "DATASET_1"

print("=" * 70)
print("DATASET_0")
print("=" * 70)

if dataset0_root.exists():
    items0 = list(dataset0_root.iterdir())
    print("Direct items:", len(items0))

    for item in items0[:30]:
        print("[FOLDER]" if item.is_dir() else "[FILE]  ", item.name)
else:
    print("❌ DATASET_0 not found")

print("\n" + "=" * 70)
print("DATASET_1")
print("=" * 70)

if dataset1_root_2.exists():
    items1 = list(dataset1_root_2.iterdir())
    print("Direct items:", len(items1))

    for item in items1[:30]:
        print("[FOLDER]" if item.is_dir() else "[FILE]  ", item.name)
else:
    print("❌ DATASET_1 not found")

DATASET_0
Direct items: 1
[FOLDER] DATASET_0

DATASET_1
Direct items: 1
[FOLDER] DATASET_1


In [19]:
# ============================================================
# STEP 18 — Audit all 8 datasets for the 4 cancer types
# ============================================================

from pathlib import Path
import pandas as pd
import re

DATASET_PATHS = {
    "DermaCon-IN-1":
        Path("/kaggle/input/datasets/abhig888/dermacon-in"),

    "DermaCon-IN-2":
        Path("/kaggle/input/datasets/hiro002/dermacon-in-dataset"),

    "ISIC-labelled":
        Path("/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled"),

    "MCC-Dataset":
        Path("/kaggle/input/datasets/quantumcoders05/mcc-dataset"),

    "Skin-Cancer-9":
        Path("/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"),

    "HAM10000":
        Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"),

    "Skin-DS":
        Path("/kaggle/input/datasets/ahmedxc4/skin-ds"),

    "All-ISIC":
        Path("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629")
}

TARGET_PATTERNS = {
    "Melanoma": [
        "melanoma",
        "mel"
    ],
    "BCC": [
        "basal cell carcinoma",
        "basal-cell carcinoma",
        "basalcell",
        "basal cell",
        "bcc"
    ],
    "SCC": [
        "squamous cell carcinoma",
        "squamous-cell carcinoma",
        "squamous cell",
        "scc",
        "akiec"
    ],
    "MCC": [
        "merkel cell carcinoma",
        "merkel-cell carcinoma",
        "merkel cell",
        "merkel",
        "mcc"
    ]
}

def detect_cancer_label(text):
    text = str(text).lower().strip()

    # Check more specific names first
    if "merkel" in text:
        return "MCC"

    if "basal cell" in text or "basalcell" in text:
        return "BCC"

    if "squamous cell" in text or "squamous-cell" in text:
        return "SCC"

    if "melanoma" in text:
        return "Melanoma"

    # Common abbreviated labels
    if text in ["mel"]:
        return "Melanoma"

    if text in ["bcc"]:
        return "BCC"

    if text in ["scc", "akiec"]:
        return "SCC"

    if text in ["mcc", "merkel"]:
        return "MCC"

    return None


# ------------------------------------------------------------
# Scan CSV metadata files
# ------------------------------------------------------------

results = []

for dataset_name, root in DATASET_PATHS.items():

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    print("Path:", root)
    print("Exists:", root.exists())

    if not root.exists():
        print("❌ Dataset not found")
        continue

    csv_files = list(root.rglob("*.csv"))

    print("CSV files found:", len(csv_files))

    dataset_counts = {
        "Melanoma": 0,
        "BCC": 0,
        "SCC": 0,
        "MCC": 0
    }

    for csv_path in csv_files:

        try:
            temp = pd.read_csv(csv_path, low_memory=False)
        except Exception:
            continue

        for col in temp.columns:

            # Only inspect columns likely to contain labels
            col_lower = str(col).lower()

            if any(x in col_lower for x in [
                "diagn", "disease", "label", "class", "dx"
            ]):

                values = temp[col].dropna().astype(str)

                for value in values:
                    detected = detect_cancer_label(value)

                    if detected:
                        dataset_counts[detected] += 1

    print("Melanoma :", dataset_counts["Melanoma"])
    print("BCC      :", dataset_counts["BCC"])
    print("SCC      :", dataset_counts["SCC"])
    print("MCC      :", dataset_counts["MCC"])

    results.append({
        "Dataset": dataset_name,
        "Melanoma": dataset_counts["Melanoma"],
        "BCC": dataset_counts["BCC"],
        "SCC": dataset_counts["SCC"],
        "MCC": dataset_counts["MCC"]
    })


# ------------------------------------------------------------
# Final summary
# ------------------------------------------------------------

summary = pd.DataFrame(results)

print("\n" + "=" * 70)
print("FINAL TYPE DATASET AUDIT")
print("=" * 70)

display(summary)


DermaCon-IN-1
Path: /kaggle/input/datasets/abhig888/dermacon-in
Exists: True
CSV files found: 3
Melanoma : 4
BCC      : 20
SCC      : 10
MCC      : 0

DermaCon-IN-2
Path: /kaggle/input/datasets/hiro002/dermacon-in-dataset
Exists: True
CSV files found: 3
Melanoma : 4
BCC      : 19
SCC      : 11
MCC      : 0

ISIC-labelled
Path: /kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled
Exists: True
CSV files found: 0
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

MCC-Dataset
Path: /kaggle/input/datasets/quantumcoders05/mcc-dataset
Exists: False
❌ Dataset not found

Skin-Cancer-9
Path: /kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic
Exists: True
CSV files found: 0
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

HAM10000
Path: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
Exists: True
CSV files found: 5
Melanoma : 1113
BCC      : 514
SCC      : 327
MCC      : 0

Skin-DS
Path: /kaggle/input/datasets/ahmedxc4/skin-ds
Exists: True
CSV fi

,Dataset,Melanoma,BCC,SCC,MCC
0,DermaCon-IN-1,4,20,10,0
1,DermaCon-IN-2,4,19,11,0
2,ISIC-labelled,0,0,0,0
3,Skin-Cancer-9,0,0,0,0
4,HAM10000,1113,514,327,0
5,Skin-DS,0,0,0,0
6,All-ISIC,8465,4921,1372,0


In [20]:
from pathlib import Path
import pandas as pd
import re

DATASET_PATHS = {
    "DermaCon-IN-1": Path("/kaggle/input/datasets/abhig888/dermacon-in"),
    "DermaCon-IN-2": Path("/kaggle/input/datasets/hiro002/dermacon-in-dataset"),
    "ISIC-labelled": Path("/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled"),
    "MCC-Dataset": Path("/kaggle/input/datasets/quantumcoders04/mcc-dataset"),
    "Skin-Cancer-9": Path("/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"),
    "HAM10000": Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"),
    "Skin-DS": Path("/kaggle/input/datasets/ahmedxc4/skin-ds"),
    "All-ISIC": Path("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629")
}

def detect_cancer_label(value):
    if pd.isna(value):
        return None

    s = str(value).strip().lower()

    # MCC
    if (
        "merkel cell" in s
        or "merkel-cell" in s
        or "merkelcell" in s
        or s == "mcc"
        or "mcc" in s
    ):
        return "MCC"

    # BCC
    if (
        "basal cell" in s
        or "basal-cell" in s
        or "basalcell" in s
        or s == "bcc"
    ):
        return "BCC"

    # SCC
    if (
        "squamous cell" in s
        or "squamous-cell" in s
        or "squamouscell" in s
        or s == "scc"
        or "actinic keratos" in s
        or "bowen" in s
    ):
        return "SCC"

    # Melanoma
    if "melanoma" in s or s == "mel":
        return "Melanoma"

    return None


def find_csv_files(root):
    if not root.exists():
        return []
    return list(root.rglob("*.csv"))


final_results = []

for dataset_name, root in DATASET_PATHS.items():

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)
    print("Path:", root)
    print("Exists:", root.exists())

    if not root.exists():
        print("❌ Dataset not found")
        final_results.append({
            "Dataset": dataset_name,
            "Melanoma": 0,
            "BCC": 0,
            "SCC": 0,
            "MCC": 0
        })
        continue

    csv_files = find_csv_files(root)
    print("CSV files found:", len(csv_files))

    counts = {
        "Melanoma": 0,
        "BCC": 0,
        "SCC": 0,
        "MCC": 0
    }

    # Scan CSV files
    for csv_path in csv_files:
        try:
            df = pd.read_csv(csv_path, low_memory=False)

            for col in df.columns:

                # Only likely label/diagnosis columns
                col_lower = str(col).lower()

                if not any(k in col_lower for k in [
                    "label", "diagnos", "disease", "class",
                    "target", "type", "condition"
                ]):
                    continue

                for value in df[col].dropna():
                    cancer = detect_cancer_label(value)

                    if cancer is not None:
                        counts[cancer] += 1

        except Exception as e:
            print("⚠️ Could not read:", csv_path.name)
            print("   Error:", str(e)[:150])

    # Also inspect folder names for datasets without CSV
    for img_path in root.rglob("*"):
        if img_path.is_file():
            parts = [p.lower() for p in img_path.parts]

            joined = " ".join(parts)

            cancer = detect_cancer_label(joined)

            if cancer is not None:
                # Folder-based detection is only useful for MCC/private datasets
                # without metadata. We do NOT add these counts here yet.
                pass

    print("Melanoma :", counts["Melanoma"])
    print("BCC      :", counts["BCC"])
    print("SCC      :", counts["SCC"])
    print("MCC      :", counts["MCC"])

    final_results.append({
        "Dataset": dataset_name,
        **counts
    })


# Final table
final_audit = pd.DataFrame(final_results)

print("\n" + "=" * 70)
print("FINAL TYPE DATASET AUDIT")
print("=" * 70)

display(final_audit)


DermaCon-IN-1
Path: /kaggle/input/datasets/abhig888/dermacon-in
Exists: True
CSV files found: 3
Melanoma : 4
BCC      : 20
SCC      : 12
MCC      : 0

DermaCon-IN-2
Path: /kaggle/input/datasets/hiro002/dermacon-in-dataset
Exists: True
CSV files found: 3
Melanoma : 4
BCC      : 19
SCC      : 13
MCC      : 0

ISIC-labelled
Path: /kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled
Exists: True
CSV files found: 0
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

MCC-Dataset
Path: /kaggle/input/datasets/quantumcoders04/mcc-dataset
Exists: True
CSV files found: 0
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

Skin-Cancer-9
Path: /kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic
Exists: True
CSV files found: 0
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

HAM10000
Path: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
Exists: True
CSV files found: 5
Melanoma : 0
BCC      : 0
SCC      : 0
MCC      : 0

Skin-DS
Path: /kaggle/input/da

,Dataset,Melanoma,BCC,SCC,MCC
0,DermaCon-IN-1,4,20,12,0
1,DermaCon-IN-2,4,19,13,0
2,ISIC-labelled,0,0,0,0
3,MCC-Dataset,0,0,0,0
4,Skin-Cancer-9,0,0,0,0
5,HAM10000,0,0,0,0
6,Skin-DS,0,0,0,0
7,All-ISIC,8673,4921,2739,0


In [21]:
# ============================================================
# STEP 19 — Verify exact cancer labels in important datasets
# ============================================================

from pathlib import Path
import pandas as pd

CHECK_DATASETS = {
    "HAM10000":
        Path("/kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000"),

    "All-ISIC":
        Path("/kaggle/input/datasets/tomooinubushi/all-isic-data-20240629"),

    "MCC-Dataset":
        Path("/kaggle/input/datasets/quantumcoders04/mcc-dataset"),

    "Skin-Cancer-9":
        Path("/kaggle/input/datasets/nodoubttome/skin-cancer9-classesisic"),

    "Skin-DS":
        Path("/kaggle/input/datasets/ahmedxc4/skin-ds"),

    "ISIC-labelled":
        Path("/kaggle/input/datasets/riyaelizashaju/isic-skin-disease-image-dataset-labelled")
}


# ------------------------------------------------------------
# 1. HAM10000 exact label distribution
# ------------------------------------------------------------

print("=" * 70)
print("HAM10000 — EXACT LABEL CHECK")
print("=" * 70)

ham_root = CHECK_DATASETS["HAM10000"]

for csv_file in ham_root.rglob("*.csv"):

    try:
        df = pd.read_csv(csv_file, low_memory=False)
    except:
        continue

    print("\nCSV:", csv_file.name)
    print("Columns:", list(df.columns))

    for col in df.columns:

        if str(col).lower() in [
            "dx",
            "diagnosis",
            "diagnosis_1",
            "disease",
            "label",
            "class"
        ]:

            print("\nLabel column:", col)
            print(df[col].value_counts(dropna=False).head(30))


# ------------------------------------------------------------
# 2. All-ISIC label columns
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ALL-ISIC — LABEL COLUMN CHECK")
print("=" * 70)

isic_root = CHECK_DATASETS["All-ISIC"]

for csv_file in isic_root.rglob("*.csv"):

    try:
        df = pd.read_csv(csv_file, low_memory=False)
    except:
        continue

    print("\nCSV:", csv_file.name)
    print("Shape:", df.shape)
    print("Columns:")
    print(list(df.columns))

    for col in df.columns:

        col_lower = str(col).lower()

        if any(x in col_lower for x in [
            "dx",
            "diagnos",
            "disease",
            "label",
            "class"
        ]):

            print("\nPossible label column:", col)
            print(df[col].value_counts(dropna=False).head(30))


# ------------------------------------------------------------
# 3. MCC DATASET — folder structure
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("MCC-DATASET — FOLDER STRUCTURE")
print("=" * 70)

mcc_root = CHECK_DATASETS["MCC-Dataset"]

print("Path:", mcc_root)
print("Exists:", mcc_root.exists())

if mcc_root.exists():

    all_items = list(mcc_root.rglob("*"))

    folders = [p for p in all_items if p.is_dir()]
    files = [p for p in all_items if p.is_file()]

    print("Total folders:", len(folders))
    print("Total files :", len(files))

    print("\nFolders:")
    for p in folders[:100]:
        print("[FOLDER]", p.relative_to(mcc_root))

    print("\nFirst 50 files:")
    for p in files[:50]:
        print("[FILE]", p.relative_to(mcc_root))


# ------------------------------------------------------------
# 4. Other two datasets — structure
# ------------------------------------------------------------

for dataset_name in ["Skin-Cancer-9", "Skin-DS", "ISIC-labelled"]:

    root = CHECK_DATASETS[dataset_name]

    print("\n" + "=" * 70)
    print(dataset_name)
    print("=" * 70)

    print("Path:", root)
    print("Exists:", root.exists())

    if root.exists():

        items = list(root.rglob("*"))

        folders = [p for p in items if p.is_dir()]
        files = [p for p in items if p.is_file()]

        print("Folders:", len(folders))
        print("Files  :", len(files))

        print("\nTop-level items:")

        for p in list(root.iterdir())[:50]:
            print(
                "[FOLDER]" if p.is_dir() else "[FILE]  ",
                p.name
            )

HAM10000 — EXACT LABEL CHECK

CSV: hmnist_8_8_RGB.csv
Columns: ['pixel0000', 'pixel0001', 'pixel0002', 'pixel0003', 'pixel0004', 'pixel0005', 'pixel0006', 'pixel0007', 'pixel0008', 'pixel0009', 'pixel0010', 'pixel0011', 'pixel0012', 'pixel0013', 'pixel0014', 'pixel0015', 'pixel0016', 'pixel0017', 'pixel0018', 'pixel0019', 'pixel0020', 'pixel0021', 'pixel0022', 'pixel0023', 'pixel0024', 'pixel0025', 'pixel0026', 'pixel0027', 'pixel0028', 'pixel0029', 'pixel0030', 'pixel0031', 'pixel0032', 'pixel0033', 'pixel0034', 'pixel0035', 'pixel0036', 'pixel0037', 'pixel0038', 'pixel0039', 'pixel0040', 'pixel0041', 'pixel0042', 'pixel0043', 'pixel0044', 'pixel0045', 'pixel0046', 'pixel0047', 'pixel0048', 'pixel0049', 'pixel0050', 'pixel0051', 'pixel0052', 'pixel0053', 'pixel0054', 'pixel0055', 'pixel0056', 'pixel0057', 'pixel0058', 'pixel0059', 'pixel0060', 'pixel0061', 'pixel0062', 'pixel0063', 'pixel0064', 'pixel0065', 'pixel0066', 'pixel0067', 'pixel0068', 'pixel0069', 'pixel0070', 'pixel0071', 

In [22]:
from pathlib import Path

MCC_ROOT = Path("/kaggle/input/datasets/quantumcoders04/mcc-dataset")

print("=" * 70)
print("MCC DATASET — STRUCTURE CHECK")
print("=" * 70)

print("Exists:", MCC_ROOT.exists())
print("Path:", MCC_ROOT)

if MCC_ROOT.exists():
    print("\nTop-level items:")
    for p in sorted(MCC_ROOT.iterdir()):
        kind = "FOLDER" if p.is_dir() else "FILE"
        print(f"[{kind}] {p.name}")

    images = [
        p for p in MCC_ROOT.rglob("*")
        if p.is_file() and p.suffix.lower() in
        (".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp")
    ]

    print("\nTotal image files:", len(images))

    print("\nFirst 30 image paths:")
    for p in images[:30]:
        print(" ", p.relative_to(MCC_ROOT))
else:
    print("ERROR: MCC dataset path does not exist.")

MCC DATASET — STRUCTURE CHECK
Exists: True
Path: /kaggle/input/datasets/quantumcoders04/mcc-dataset

Top-level items:
[FOLDER] mcc image

Total image files: 17

First 30 image paths:
  mcc image/41698_2025_987_Fig1_HTML.webp
  mcc image/imge.jpeg
  mcc image/canvas 1.png
  mcc image/images_large_rg.2019190102.fig2.jpeg
  mcc image/12885_2013_Article_4464_Fig1_HTML.webp
  mcc image/images_g02mr14c1x.jpeg
  mcc image/cancers-16-03586-g001.png
  mcc image/canvas.png
  mcc image/full-1255fig02.jpeg
  mcc image/cancers-16-03586-g002.png
  mcc image/41571_2018_103_Fig1_HTML.jpg
  mcc image/cup13910-fig-0001-m.jpg
  mcc image/canvas 2.png
  mcc image/5cm.webp
  mcc image/12885_2013_Article_4464_Fig3_HTML.webp
  mcc image/cup13910-fig-0001-m - Copy.jpg
  mcc image/images_g02mr14c2x.jpeg
